# 01 - Exploracao dos dados

Objetivo: entender a estrutura do dataset IMDb, a quantidade de reviews por classe e exemplos reais de textos positivos e negativos.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "aclImdb"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

print("Projeto:", PROJECT_ROOT)
print("Dados:", DATA_DIR)


Projeto: C:\Users\Carlos henrique\Documents\deeplearning
Dados: C:\Users\Carlos henrique\Documents\deeplearning\data\raw\aclImdb


In [2]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from src.imdb_data import count_reviews, load_reviews

sns.set_theme(style="whitegrid")


## Quantidade de reviews por conjunto e classe

In [3]:
counts = count_reviews(DATA_DIR)
counts_df = (
    pd.DataFrame(counts)
    .T
    .rename_axis("split")
    .reset_index()
)
counts_df["total"] = counts_df[["neg", "pos"]].sum(axis=1)
counts_df


,split,neg,pos,total
0,train,12500,12500,25000
1,test,12500,12500,25000


In [4]:
plot_df = counts_df.melt(id_vars="split", value_vars=["neg", "pos"], var_name="classe", value_name="reviews")
ax = sns.barplot(data=plot_df, x="split", y="reviews", hue="classe")
ax.set_title("Reviews por conjunto e classe")
ax.set_xlabel("Conjunto")
ax.set_ylabel("Quantidade de reviews")
plt.show()


C:\Users\Carlos henrique\AppData\Local\Temp\ipykernel_28744\3761704643.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Carregamento em DataFrame (amostra)

In [5]:
from src.imdb_data import iter_review_files

rows = []
for split in ("train", "test"):
    for label_name in ("neg", "pos"):
        files = sorted((DATA_DIR / split / label_name).glob("*.txt"))[:500]
        for path in files:
            rows.append({
                "split": split,
                "label": label_name,
                "target": 1 if label_name == "pos" else 0,
                "review": path.read_text(encoding="utf-8"),
                "file": str(path),
            })

df = pd.DataFrame(rows)
df["num_chars"] = df["review"].str.len()
df["num_words"] = df["review"].str.split().str.len()

print(df.shape)
df.head()

(2000, 7)


,split,label,target,review,file,num_chars,num_words
0,train,neg,0,Story of a man who has unnatural feelings for ...,C:\Users\Carlos henrique\Documents\deeplearnin...,655,112
1,train,neg,0,Airport '77 starts as a brand new luxury 747 p...,C:\Users\Carlos henrique\Documents\deeplearnin...,4466,801
2,train,neg,0,This film lacked something I couldn't put my f...,C:\Users\Carlos henrique\Documents\deeplearnin...,807,141
3,train,neg,0,"Sorry everyone,,, I know this is supposed to b...",C:\Users\Carlos henrique\Documents\deeplearnin...,862,154
4,train,neg,0,When I was little my parents took me along to ...,C:\Users\Carlos henrique\Documents\deeplearnin...,2326,395


In [6]:
df.groupby(["split", "label"])[["num_chars", "num_words"]].agg(["count", "mean", "median", "min", "max"]).round(2)


num_chars                              num_words                 \
                count     mean  median  min    max     count    mean median   
split label                                                                   
test  neg         500  1240.44   914.5  162   5796       500  220.39  164.5   
      pos         500  1312.89   953.0  192   6007       500  230.46  169.5   
train neg         500  1270.82   983.5  106   5758       500  225.58  177.0   
      pos         500  1384.57  1037.0  169  10321       500  244.03  183.5   

                       
            min   max  
split label            
test  neg    32   989  
      pos    32   997  
train neg    19  1002  
      pos    30  1839

In [7]:
ax = sns.histplot(data=df, x="num_words", hue="label", bins=60, kde=True)
ax.set_title("Distribuicao do tamanho das reviews (amostra de 2.000)")
ax.set_xlabel("Numero de palavras")
plt.show()


C:\Users\Carlos henrique\AppData\Local\Temp\ipykernel_28744\482695971.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Exemplos de reviews

In [8]:
for label in ["neg", "pos"]:
    exemplo = df[df["label"] == label].sample(1, random_state=42).iloc[0]
    print(f"Classe: {label}")
    print(f"Arquivo: {exemplo['file']}")
    print(exemplo["review"][:900].replace("<br />", " "))
    print("-" * 100)

Classe: neg
Arquivo: C:\Users\Carlos henrique\Documents\deeplearning\data\raw\aclImdb\test\neg\10019_1.txt
This is truly, without exaggerating, one of the worst Slasher movies ever made. I know, it came out in the 80's following a tendency started by "Friday the 13th". "The Prey" copies the fore-mentioned movie in many aspects. The woods setting, the killer, the dumb teens, the gore, etc.  But "The Prey" is as bad as you might expect. I didn't even remember about it if it wasn't for coincidence.  Well, the killer is in fact human so don't expect a supernatural killer in the likes of Jason. The situations rather boring and lack of tension, gore, violence, etc. It just does not works for a slasher flick.  The acting is simply horrid. The score is horrible! a combination of boring instruments with cheesy 80's tunes?! I won't even mention the technical aspects of the movie because believe me, it seems that it cost only 20 dollars.  Please avoid this one lik
--------------------------------

## Conclusao

O dataset esta balanceado entre reviews positivas e negativas, com divisao pronta em treino e teste. Como os textos variam bastante de tamanho, o proximo notebook aplica vetorizacao e padding antes do treinamento.